# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshitttt077/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Author:** Harshit Kudhial (`@harshitttt077`) | **Track:** Machine Learning | **Lane:** Lane 2 Refresh Scoring


## 1. Unit of analysis + time window
- **Unit of Analysis:** Exactly one row = one unique content asset (`content_id`) published under an enterprise client domain (`client_id`).
- **Time Window:** 90-day trailing aggregation window for search impressions, clicks, sessions, and engagement.
- **Grain Verification:** Guaranteed unique on `content_id`.


In [1]:
import pandas as pd, numpy as np
df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')
print(f"Total Rows: {len(df):,} | Unique Content IDs: {df['content_id'].nunique():,}")
assert len(df) == df['content_id'].nunique(), "Grain violation: duplicate content_ids detected!"
print("Grain Assert Passed: Exactly 1 row per unique content_id.")


Total Rows: 30,000 | Unique Content IDs: 30,000
Grain Assert Passed: Exactly 1 row per unique content_id.


## 2. Fields: feature / label / context / excluded
- **Observable Features:** `content_age_days`, `days_since_last_update`, `impressions_90d`, `clicks_90d`, `sessions_90d`, `avg_position`, `ctr`, `word_count`, `engagement_rate`, `scroll_rate`.
- **Target Label:** `is_declining_label` derived from `trend_direction == 'down'`.
- **Context/Grouping:** `client_id` (used strictly for grouped client splits).
- **Excluded Fields:** `trend_pct` and `trend_direction` (leakage risks), customer PII, raw URLs.


In [2]:
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
excluded = ['trend_pct', 'trend_direction', 'client_id', 'content_id']
assert all(f not in features for f in excluded), "Data Contract violation: excluded columns present in features!"
print(f"Features: {features}\nExcluded: {excluded}\nAssertion verified.")


Features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Excluded: ['trend_pct', 'trend_direction', 'client_id', 'content_id']
Assertion verified.


## 3. Verify it with queries (grain, counts, missing values, windows)
We query null rates, class balance, and summary distributions across the 30,000 rows.


In [3]:
null_counts = df[features].isna().sum()
print("Missing value counts across candidate features:")
print(null_counts)
print(f"\nDocumented in Contract: word_count has {null_counts['word_count']:,} missing values, filled with 0.")
df_clean = df[features].fillna(0)
assert df_clean.isna().sum().max() == 0, "Unexpected null values remain after contract fill!"
print(f"\nClass balance of trend_direction:\n{df['trend_direction'].value_counts(normalize=True)*100}")


Missing value counts across candidate features:
content_age_days             0
days_since_last_update       0
impressions_90d              0
avg_position                 0
ctr                          0
word_count                7699
dtype: int64

Documented in Contract: word_count has 7,699 missing values, filled with 0.

Class balance of trend_direction:
trend_direction
down      54.206667
stable    19.873333
up        14.626667
new        7.453333
flat       3.840000
Name: proportion, dtype: float64


## 4. Data limits
- **Limits:** Observational cross-sectional cut; cannot infer causal lift from content rewrites without an A/B test. Macro algorithm shifts (AI Overviews) are unobserved.


In [4]:
print("Data Contract Summary: 30,000 rows verified, zero nulls in core features, zero PII, ready for feature engineering.")


Data Contract Summary: 30,000 rows verified, zero nulls in core features, zero PII, ready for feature engineering.
